In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F



In [2]:

# ── 1. VAE Decoder ─────────────────────────────────────────────────────────────
# Takes a latent vector z (sampled via reparameterisation) and reconstructs
# the upper-triangle of the adjacency matrix for a graph of N nodes.
#
# Flow:
#   z  →  MLP  →  (N*N,)  →  sigmoid  →  edge probabilities  →  A_hat (N×N)
#
# We predict a FIXED maximum number of nodes (max_nodes) and during sampling
# we threshold the probabilities to get a binary adjacency matrix.

class GNNDecoder(nn.Module):
    """
    MLP decoder for a Graph VAE.

    Takes a latent vector z and outputs an adjacency matrix of shape (N, N).

    Input:
        z : (batch_size, latent_dim)  — sampled latent from reparameterisation

    Output:
        A_logits : (batch_size, max_nodes, max_nodes)  — raw edge logits (pre-sigmoid)
    """

    def __init__(
        self,
        latent_dim: int = 32,
        hidden_dim: int = 128,
        max_nodes: int = 28,       # MUTAG max graph size
    ):
        super().__init__()

        self.max_nodes = max_nodes
        output_dim = max_nodes * max_nodes   # full adjacency matrix (flattened)

        self.mlp = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, z):
        """
        Args:
            z : (batch_size, latent_dim)

        Returns:
            A_logits : (batch_size, max_nodes, max_nodes)
        """
        out = self.mlp(z)                                        # (B, N*N)
        A_logits = out.view(-1, self.max_nodes, self.max_nodes)  # (B, N, N)

        # Symmetrise: real adjacency matrices are symmetric
        A_logits = (A_logits + A_logits.transpose(1, 2)) / 2

        return A_logits



In [3]:

# ── 2. Reparameterisation trick ────────────────────────────────────────────────
# Instead of sampling z ~ N(mean, var) directly (not differentiable),
# we sample eps ~ N(0,1) and compute z = mean + eps * std.
# This keeps gradients flowing through mean and log_var.

def reparameterise(z_mean, z_log_var):
    """
    Args:
        z_mean    : (batch_size, latent_dim)
        z_log_var : (batch_size, latent_dim)

    Returns:
        z : (batch_size, latent_dim)  — sampled latent vector
    """
    std = torch.exp(0.5 * z_log_var)           # convert log_var → std
    eps = torch.randn_like(std)                 # sample noise ~ N(0, I)
    return z_mean + eps * std                   # reparameterised sample



In [4]:

# ── 3. VAE Loss ────────────────────────────────────────────────────────────────
# Total VAE loss = Reconstruction loss + KL divergence
#
# Reconstruction: BCE between predicted edge probs and true adjacency matrix
#   L_recon = BCE( sigmoid(A_logits), A_true )
#
# KL divergence: measures how far q(z|x) is from prior N(0,I)
#   L_KL = -0.5 * sum(1 + log_var - mean^2 - exp(log_var))
#
# beta controls the trade-off (beta=1 is standard VAE)

def vae_loss(A_logits, A_target, z_mean, z_log_var, beta=1.0):
    """
    Args:
        A_logits   : (batch_size, max_nodes, max_nodes)  — decoder output (logits)
        A_target   : (batch_size, max_nodes, max_nodes)  — true adjacency matrix
        z_mean     : (batch_size, latent_dim)
        z_log_var  : (batch_size, latent_dim)
        beta       : KL weight (default 1.0 = standard VAE)

    Returns:
        total_loss : scalar
        recon_loss : scalar (for logging)
        kl_loss    : scalar (for logging)
    """
    batch_size = A_logits.shape[0]

    # Mask diagonal (no self-loops) — exclude from loss entirely
    n = A_logits.shape[1]
    diag_mask = torch.eye(n, device=A_logits.device).bool().unsqueeze(0)  # (1, N, N)
    A_logits  = A_logits.masked_fill(diag_mask, 0.0)
    A_target  = A_target.masked_fill(diag_mask, 0.0)

    # Reconstruction loss — treat each edge as independent Bernoulli
    recon_loss = F.binary_cross_entropy_with_logits(
        A_logits,
        A_target,
        reduction='sum'
    ) / batch_size

    # KL divergence — analytical formula for Gaussian
    kl_loss = -0.5 * torch.sum(
        1 + z_log_var - z_mean.pow(2) - z_log_var.exp()
    ) / batch_size

    total_loss = recon_loss + beta * kl_loss

    return total_loss, recon_loss, kl_loss



In [5]:


# ── 4. Helper: pad adjacency matrix to max_nodes ──────────────────────────────
# Training graphs have variable sizes; we pad smaller ones with zeros.

def pyg_to_dense_adj(data, max_nodes):
    """
    Convert a PyG Data object to a dense (max_nodes x max_nodes) adjacency matrix.

    Args:
        data      : PyG Data object (single graph)
        max_nodes : int — size to pad to

    Returns:
        A : (max_nodes, max_nodes) float tensor
    """
    A = torch.zeros(max_nodes, max_nodes)
    src, dst = data.edge_index
    # Only keep edges within max_nodes (safety check)
    mask = (src < max_nodes) & (dst < max_nodes)
    A[src[mask], dst[mask]] = 1.0
    return A



In [7]:

# ── 5. Sanity check ────────────────────────────────────────────────────────────
if __name__ == "__main__":
    from torch_geometric.datasets import TUDataset
    from gnn_encoder import GINEncoder
    from torch_geometric.loader import DataLoader

    dataset = TUDataset(root="data/TUDataset", name="MUTAG")
    loader  = DataLoader(dataset[:32], batch_size=8, shuffle=False)

    MAX_NODES  = 28
    LATENT_DIM = 32

    encoder = GINEncoder(input_dim=dataset.num_node_features, latent_dim=LATENT_DIM)
    decoder = GNNDecoder(latent_dim=LATENT_DIM, max_nodes=MAX_NODES)

    print(decoder)
    print(f"Trainable parameters: {sum(p.numel() for p in decoder.parameters()):,}")

    batch = next(iter(loader))

    # Encode
    z_mean, z_log_var = encoder(batch.x.float(), batch.edge_index, batch.batch)

    # Reparameterise
    z = reparameterise(z_mean, z_log_var)
    print(f"\nz shape: {z.shape}")                  # (8, 32)

    # Decode
    A_logits = decoder(z)
    print(f"A_logits shape: {A_logits.shape}")      # (8, 28, 28)

    # Build target adjacency matrices for this batch
    A_target = torch.stack([
        pyg_to_dense_adj(dataset[i], MAX_NODES)
        for i in range(8)
    ])
    print(f"A_target shape: {A_target.shape}")      # (8, 28, 28)

    # Compute loss
    loss, recon, kl = vae_loss(A_logits, A_target, z_mean, z_log_var)
    print(f"\nTotal loss : {loss.item():.4f}")
    print(f"Recon loss : {recon.item():.4f}")
    print(f"KL loss    : {kl.item():.4f}")


GNNDecoder(
  (mlp): Sequential(
    (0): Linear(in_features=32, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=784, bias=True)
  )
)
Trainable parameters: 121,872

z shape: torch.Size([8, 32])
A_logits shape: torch.Size([8, 28, 28])
A_target shape: torch.Size([8, 28, 28])

Total loss : 546.4795
Recon loss : 544.8314
KL loss    : 1.6481
